In [1]:
# Cell 1: Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
# Cell 2: Load Data
data = pd.read_csv("..\data\processed\cleaned.csv")

print(data.shape)
data.head()

(7043, 46)


<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Niku\AppData\Local\Temp\ipykernel_39644\3666073078.py:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  data = pd.read_csv("..\data\processed\cleaned.csv")


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,...,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn
0,0,1,29.85,29.85,True,False,False,True,True,False,...,True,False,False,False,True,False,False,True,False,0
1,0,34,56.95,1889.50,False,True,True,False,True,False,...,False,True,False,True,False,False,False,False,True,0
2,0,2,53.85,108.15,False,True,True,False,True,False,...,True,False,False,False,True,False,False,False,True,1
3,0,45,42.30,1840.75,False,True,True,False,True,False,...,False,True,False,True,False,True,False,False,False,0
4,0,2,70.70,151.65,True,False,True,False,True,False,...,True,False,False,False,True,False,False,True,False,1


In [4]:
# Cell 3: Split

X = data.drop(columns=["Churn"])
y = data["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(5634, 45) (1409, 45)


In [5]:
# Cell 4: Drop redundant columns

cols_to_drop = [
    "gender_Female",
    "Partner_No",
    "Dependents_No",
    "PhoneService_No",
    "PaperlessBilling_No",
    "Contract_One year",
    "PaymentMethod_Mailed check"
]

X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

In [6]:
# Cell 5: Core features

def add_core_features(df):
    df = df.copy()

    # Spend efficiency
    df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)

    # Charge intensity
    df["ChargePerTenure"] = df["MonthlyCharges"] / (df["tenure"] + 1)

    # Customer lifetime value proxy
    df["CLV_proxy"] = df["tenure"] * df["MonthlyCharges"]

    return df

X_train = add_core_features(X_train)
X_test = add_core_features(X_test)

In [7]:
# Cell 6: Risk flags

def add_risk_flags(df):
    df = df.copy()

    df["IsMonthToMonth"] = df["Contract_Month-to-month"]
    df["IsFiber"] = df["InternetService_Fiber optic"]
    df["IsElectronicPayment"] = df["PaymentMethod_Electronic check"]

    return df

X_train = add_risk_flags(X_train)
X_test = add_risk_flags(X_test)

In [8]:
# Cell 7: Interactions

def add_interactions(df):
    df = df.copy()

    df["MM_Paperless"] = df["Contract_Month-to-month"] * df["PaperlessBilling_Yes"]
    df["Fiber_HighCharge"] = df["InternetService_Fiber optic"] * (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    df["Tenure_Charge"] = df["tenure"] * df["MonthlyCharges"]

    return df

X_train = add_interactions(X_train)
X_test = add_interactions(X_test)

In [9]:
# Cell 8: Nonlinear transforms

def add_transforms(df):
    df = df.copy()

    df["log_tenure"] = np.log1p(df["tenure"])
    df["log_totalcharges"] = np.log1p(df["TotalCharges"])

    return df

X_train = add_transforms(X_train)
X_test = add_transforms(X_test)

In [10]:
# Cell 9: Binning

def add_bins(df):
    df = df.copy()

    df["TenureGroup"] = pd.cut(df["tenure"], bins=[0,12,24,48,72], labels=[0,1,2,3])
    df["ChargeGroup"] = pd.qcut(df["MonthlyCharges"], q=4, labels=False, duplicates="drop")

    return df

X_train = add_bins(X_train)
X_test = add_bins(X_test)

In [11]:
# Cell 10: Fill NaNs

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

In [12]:
# Cell 11: Scaling (optional for XGBoost)

scaler = StandardScaler()

num_cols = X_train.select_dtypes(include=np.number).columns

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [13]:
# Cell 12: Final dataset check

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

X_train.head()

Train shape: (5634, 51)
Test shape: (1409, 51)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,...,IsMonthToMonth,IsFiber,IsElectronicPayment,MM_Paperless,Fiber_HighCharge,Tenure_Charge,log_tenure,log_totalcharges,TenureGroup,ChargeGroup
3738,-0.441773,0.102371,-0.521976,-0.262257,True,False,False,False,False,True,...,True,False,True,False,False,-0.253464,0.468865,0.316915,2,-0.446071
3151,-0.441773,-0.711743,0.337478,-0.503635,True,True,True,True,True,False,...,True,True,False,False,True,-0.514920,-0.235332,0.067683,1,0.448611
4860,-0.441773,-0.793155,-0.809013,-0.749883,True,True,True,False,False,True,...,False,False,False,False,False,-0.778067,-0.351288,-0.358545,1,-0.446071
3867,-0.441773,-0.263980,0.284384,-0.172722,False,True,False,True,True,False,...,False,False,False,False,False,-0.170483,0.219047,0.389209,2,0.448611
3810,-0.441773,-1.281624,-0.676279,-0.989374,True,True,True,True,True,False,...,True,False,True,False,False,-0.989954,-2.041081,-1.995946,0,-0.446071
